# Ministral judge: positive and negative pilot controls

Run this notebook in **Google Colab → Runtime → Change runtime type → A100 GPU**.
A previously tested L4 may also fit; the unchanged BF16 launcher requires at least
22 GiB and BF16 support. A T4 is unsuitable for this configuration. Do not change
precision/model revision to make it fit.

Upload **`judge_pilot_code.bundle`** when prompted. This contains the exact committed
code and all fixtures/docs, so **no GitHub push, login, EC2 instance, real dataset,
or separate rubric upload is required**. Internet is still needed for Python
packages and model weights. Allow ample disk space (roughly 35 GB for weights plus
the serving environment/downloads).

This run checks **2 positive and 4 negative synthetic cases**, 20 samples each
(**120 score vectors**). It does not run the 50+50 real evaluation or human
calibration. Correlation reports will correctly have undefined values for these
small task cohorts; real-cohort correlations remain a later step.

Run cells in order. Keep the runtime open while scoring. The scoring cell stops the
local server in `finally`; an emergency cleanup cell is also provided. Disconnect
and delete the GPU runtime when finished to avoid idle usage.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, shutil, signal, subprocess, sys, time

CODE_COMMIT = "3fa4046cad6718fe449c300eb6f6c3410c436da8"
BUNDLE_SHA256 = "b33e6ef08af4042aa934cb2463f85ec527b09fcb8e90247ceb4de132196b43d5"
BUNDLE_PATH = Path("/content/judge_pilot_code.bundle")
PROJECT = Path("/content/298A_judge_pilot_3fa4046")
ENV = Path("/content/ministral_judge_env")
PYTHON = ENV / "bin/python"
BASE_URL = "http://127.0.0.1:8001/v1"
SAVE_TO_DRIVE = True
server = None
server_log = None

## 1. Upload the supplied code bundle
Select `judge_pilot_code.bundle` from the downloaded package. It is a Git bundle,
not a Python file. The checksum below checks that it is the supplied package.

In [ ]:
from google.colab import files
uploaded = files.upload()
matching = [data for data in uploaded.values()
            if hashlib.sha256(data).hexdigest() == BUNDLE_SHA256]
if not matching:
    raise ValueError("Uploaded files do not contain the expected code bundle.")
BUNDLE_PATH.write_bytes(matching[0])
del uploaded, matching
print("Bundle checksum verified (upload filename may differ).")


## 2. Restore the committed code and run local checks
No authentication is required. The checkout and protocol checks retain the original
commit identity; this is not a newly initialized copy of source files.

In [ ]:
assert hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest() == BUNDLE_SHA256
if not PROJECT.exists():
    subprocess.run(["git", "clone", str(BUNDLE_PATH), str(PROJECT)], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", CODE_COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
assert head == CODE_COMMIT, "Use a fresh PROJECT directory; this checkout is different."
test_result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_judge*.py", "-v"], cwd=PROJECT, capture_output=True, text=True)
print(test_result.stdout)
print(test_result.stderr)
if test_result.returncode:
    raise RuntimeError("Setup tests failed. Share the printed test output above before continuing.")
subprocess.run([sys.executable, "-c", "from src.evaluation.judge.run import CONFIG,RUBRIC,committed_protocol; print(committed_protocol(CONFIG,RUBRIC))"], cwd=PROJECT, check=True)
print("Pinned code and protocol verified:", head)

## 3. Create the isolated GPU environment
This keeps the serving packages separate from Colab's own packages. First-time
installation/model download can take several minutes. A valid GPU runtime is required.

In [ ]:
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU utility found. Select an L4/A100 GPU runtime and rerun from the first cell.")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError("Select a GPU runtime before continuing.")
print(gpu.stdout)
print("Free runtime disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

def environment_ready():
    if not PYTHON.exists():
        return False
    return subprocess.run([str(PYTHON), "-c", "import sys,pip; assert sys.prefix != sys.base_prefix"], capture_output=True).returncode == 0

if not environment_ready():
    bootstrap = ENV.parent / (ENV.name + "_bootstrap")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--target", str(bootstrap), "virtualenv==20.35.4"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", "--python", sys.executable, "--no-download", "--no-periodic-update", str(ENV)], env={**os.environ, "PYTHONPATH": str(bootstrap)}, check=True)
    assert environment_ready(), "Isolated Python/pip setup failed."
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(PROJECT / "src/evaluation/judge/requirements-serving.txt")], check=True)
subprocess.run([str(PYTHON), "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0)); assert torch.cuda.is_bf16_supported(); assert torch.cuda.get_device_properties(0).total_memory / 2**30 >= 22"], check=True)

## 4. Choose persistent storage
Default destination: `MyDrive/298A/geval_judge_runs/`. Every attempt uses a new
folder. Save failures as well as successes; don't overwrite the historical smoke run.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/298A/geval_judge_runs")
else:
    OUTPUT_ROOT = Path("/content/geval_judge_runs")
RUN_ID = "pilot_controls_" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "package_provenance.json").write_text(json.dumps({"code_commit": CODE_COMMIT, "bundle_sha256": BUNDLE_SHA256, "scope": "2 positive plus 4 negative synthetic cases; not real evaluation"}, indent=2) + "\n")
print("Save this folder:", RUN_DIR)

## 5. Start the judge, score both sets, and stop it
This cell runs both sets against the same protocol. It writes all 120 score vectors,
means, metadata, environment details, correlation diagnostics, and a control summary.
**Completed scoring is not proof that the negative controls behaved correctly.**
Inspect the summary in the next cell. No score expectations are sent to the judge.

In [ ]:
if (RUN_DIR / "serving").exists():
    raise RuntimeError("This attempt was already started. Rerun cell 4 for a new output folder.")

def stop_judge():
    global server_log
    if server is not None:
        # The launcher and vLLM children share the new process group.
        try:
            os.killpg(server.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            server.wait(timeout=30)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(server.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        server.wait(timeout=10)
    if server_log is not None and not server_log.closed:
        server_log.close()

try:
    server_log = (RUN_DIR / "server.log").open("w")
    server = subprocess.Popen([str(PYTHON), "-m", "src.evaluation.judge.serve", "--record-dir", str(RUN_DIR / "serving")], cwd=PROJECT, stdout=server_log, stderr=subprocess.STDOUT, start_new_session=True)
    check_code = "from src.evaluation.judge.run import Endpoint,load_config; Endpoint('http://127.0.0.1:8001/v1',timeout=10).verify(load_config())"
    deadline = time.monotonic() + 1200
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError("Judge exited. Inspect server.log in the saved folder.")
        if subprocess.run([str(PYTHON), "-c", check_code], cwd=PROJECT, capture_output=True).returncode == 0:
            print("Pinned judge alias ready.")
            break
        time.sleep(10)
    else:
        raise TimeoutError("Judge not ready after 20 minutes; inspect saved server.log.")
    summary = []
    for name, fixture in [("positive", "synthetic_cases.jsonl"), ("negative", "pilot_negative_cases.jsonl")]:
        destination = RUN_DIR / name
        subprocess.run([str(PYTHON), "-m", "src.evaluation.judge.run", "--cases", str(PROJECT / "tests/fixtures/judge" / fixture), "--output", str(destination), "--mode", "smoke", "--base-url", BASE_URL], cwd=PROJECT, check=True)
        subprocess.run([str(PYTHON), "-m", "src.evaluation.judge.analyze_pilot", "--scores", str(destination / "scores.jsonl"), "--output", str(destination / "criterion_correlations.json")], cwd=PROJECT, check=True)
        for line in (destination / "scores.jsonl").read_text().splitlines():
            record = json.loads(line)
            summary.append({"set": name, "case_id": record["case_id"], "task": record["task"], "means": record["scores"]})
    (RUN_DIR / "control_summary.json").write_text(json.dumps({"scoring_status": "COMPLETE", "behavior_review": "PENDING inspection; no automatic control acceptance threshold", "cases": summary}, indent=2) + "\n")
    print("Both scoring runs completed. Inspect their means next.")
finally:
    stop_judge()
    print("Local judge shutdown requested. Outputs:", RUN_DIR)

## 6. Inspect the control scores
- `negative_explanation_unsupported`: expect low **faithfulness**.
- `negative_profile_unsupported`: expect low **faithfulness and relevance**.
- Both `off_topic` cases: expect low **relevance**.

These answers can remain fluent/coherent. Do not require every score to be low.
If an obviously bad answer gets high scores on the affected criteria, retain the
results and investigate before the real evaluation. No numerical pass threshold
has been precommitted. The two positive cases provide context, not matched controls.

Correlations are undefined here because each separate set has fewer than three
cases per task. This is expected, not an execution failure. Real 50+50 cases and
independent human scores are still required for the planned evaluation/calibration.

In [ ]:
summary_path = RUN_DIR / "control_summary.json"
if not summary_path.exists():
    raise RuntimeError("Scoring did not complete; inspect the saved metadata/logs.")
summary = json.loads(summary_path.read_text())
for case in summary["cases"]:
    print(case["case_id"], case["means"])
print("Behavior review:", summary["behavior_review"])

## 7. Download evidence and disconnect the GPU
This ZIP contains the new synthetic score records and runtime evidence. The ZIP can
be attached for review; the executed notebook can also be downloaded using
**File → Download → Download .ipynb**. The code package is separate from this output.

In [ ]:
archive = shutil.make_archive(str(Path("/content") / RUN_ID), "zip", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print("Saved evidence archive:", archive)
from google.colab import files
files.download(archive)
print("After saving, choose Runtime → Disconnect and delete runtime.")

## Emergency cleanup (only if interrupted)
Run this if the scoring cell was interrupted. It stops this notebook's server group.
Then save any available evidence and disconnect/delete the GPU runtime.

In [ ]:
if "stop_judge" in globals():
    stop_judge()
print("Cleanup requested. Disconnect and delete the GPU runtime when finished.")